# UK Electricity Demand — Exploratory Data Analysis

Target: **National Demand (ND, MW)**, half-hourly, from the processed table
`data/processed/energy_demand_30min.parquet`.

Reusable loading/aggregation lives in `energy_forecasting.evaluation.plots`; this
notebook is for investigation and figures. Publication-quality figures are written
to `reports/figures/`. Only the 4–5 strongest belong in the README — see the final cell.

**What we are looking for:** daily / weekly / annual seasonality, higher winter
demand, holiday effects, a non-linear temperature–demand relationship, and
exceptional periods (e.g. 2020 lockdowns).


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="talk")
except Exception:
    plt.style.use("seaborn-v0_8-whitegrid") if "seaborn-v0_8-whitegrid" in plt.style.available else None

plt.rcParams.update({"figure.figsize": (13, 5), "figure.dpi": 110, "savefig.dpi": 150})

from energy_forecasting.evaluation.plots import (
    load_processed, add_calendar_columns, mean_profile, save_fig, FIGURES_DIR,
)

df = add_calendar_columns(load_processed())
df_i = df.set_index("timestamp_utc")
demand = df_i["nd_mw"]
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print(f"Rows: {len(df):,} | span: {df['timestamp_utc'].min()} -> {df['timestamp_utc'].max()}")
print(f"Figures -> {FIGURES_DIR}")
df[["nd_mw","temperature_mean","apparent_temperature_mean"]].describe()


## 1. Complete demand time series
Daily mean overlaid on the raw half-hourly series so long-range structure (annual cycle, any level shifts such as 2020) is visible.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(demand.index, demand.values, lw=0.2, alpha=0.3, color="steelblue", label="half-hourly")
daily = demand.resample("1D").mean()
ax.plot(daily.index, daily.values, lw=1.2, color="navy", label="daily mean")
ax.set_title("GB National Demand, complete series")
ax.set_ylabel("Demand (MW)"); ax.legend(loc="upper right")
save_fig(fig, "fig_01_demand_timeseries"); plt.show()


## 2. Demand by year
Distribution per calendar year — watch for the 2020 lockdown dip and any long-run downward trend (efficiency, embedded solar/wind netting off).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
years = sorted(df["year"].unique())
ax.boxplot([df.loc[df["year"]==y, "nd_mw"].dropna() for y in years], labels=years, showfliers=False)
ax.set_title("Demand distribution by year"); ax.set_ylabel("Demand (MW)"); ax.set_xlabel("Year")
save_fig(fig, "fig_02_demand_by_year"); plt.show()
df.groupby("year")["nd_mw"].agg(["mean","median","std"]).round(0)


## 3. Average demand by month
Annual seasonality: expect winter (Dec–Feb) well above summer (Jun–Aug).

In [ ]:
by_month = mean_profile(df, "month")
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(by_month.index, by_month.values, color="indianred")
ax.set_xticks(range(1,13)); ax.set_title("Average demand by month")
ax.set_ylabel("Mean demand (MW)"); ax.set_xlabel("Month")
save_fig(fig, "fig_03_demand_by_month"); plt.show()


## 4. Average demand by day of week
Weekly seasonality: weekdays higher, a clear weekend drop.

In [ ]:
by_dow = mean_profile(df, "day_of_week")
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(7), by_dow.reindex(range(7)).values, color="seagreen")
ax.set_xticks(range(7)); ax.set_xticklabels(DOW); ax.set_title("Average demand by day of week")
ax.set_ylabel("Mean demand (MW)")
save_fig(fig, "fig_04_demand_by_dayofweek"); plt.show()


## 5. Average half-hour demand profile
Daily seasonality: overnight trough, morning ramp, evening peak.

In [ ]:
by_tod = mean_profile(df, "time_of_day")
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(by_tod.index, by_tod.values, marker="o", ms=3, color="darkorange")
ax.set_title("Average within-day demand profile (48 half-hours)")
ax.set_xlabel("Hour of day (local)"); ax.set_ylabel("Mean demand (MW)"); ax.set_xticks(range(0,25,2))
save_fig(fig, "fig_05_halfhour_profile"); plt.show()


## 6. Weekday vs weekend
Same within-day profile split by weekend flag — expect lower, flatter, later weekend peaks.

In [ ]:
prof = mean_profile(df, ["is_weekend","time_of_day"]).unstack(0)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(prof.index, prof[0], label="Weekday", color="navy")
ax.plot(prof.index, prof[1], label="Weekend", color="crimson")
ax.set_title("Weekday vs weekend demand profile"); ax.set_xlabel("Hour of day (local)")
ax.set_ylabel("Mean demand (MW)"); ax.set_xticks(range(0,25,2)); ax.legend()
save_fig(fig, "fig_06_weekday_vs_weekend"); plt.show()


## 7. Holiday vs non-holiday
GB bank holidays (`is_bank_holiday`) typically look like a Sunday — lower daytime demand.

In [ ]:
prof = mean_profile(df, ["is_bank_holiday","time_of_day"]).unstack(0)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(prof.index, prof.get(0), label="Non-holiday", color="navy")
if 1 in prof.columns:
    ax.plot(prof.index, prof[1], label="Bank holiday", color="darkgreen")
ax.set_title("Holiday vs non-holiday demand profile"); ax.set_xlabel("Hour of day (local)")
ax.set_ylabel("Mean demand (MW)"); ax.set_xticks(range(0,25,2)); ax.legend()
save_fig(fig, "fig_07_holiday_vs_nonholiday"); plt.show()
df.groupby("is_bank_holiday")["nd_mw"].mean().round(0)


## 8. Demand vs temperature
Expect a non-linear, mostly decreasing relationship (heating drives winter demand), possibly flattening or rising slightly at high temperatures.

In [ ]:
def scatter_with_binned_mean(xcol, name, title):
    x = df[xcol]; y = df["nd_mw"]
    m = x.notna() & y.notna()
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.scatter(x[m], y[m], s=3, alpha=0.05, color="grey")
    bins = pd.cut(x[m], np.arange(np.floor(x[m].min()), np.ceil(x[m].max())+1, 1))
    binned = y[m].groupby(bins).mean()
    centers = [iv.mid for iv in binned.index]
    ax.plot(centers, binned.values, color="red", lw=2.5, label="binned mean")
    ax.set_title(title); ax.set_xlabel(f"{name} (°C)"); ax.set_ylabel("Demand (MW)"); ax.legend()
    return fig

fig = scatter_with_binned_mean("temperature_mean", "UK mean temperature", "Demand vs temperature")
save_fig(fig, "fig_08_demand_vs_temperature"); plt.show()


## 9. Demand vs apparent temperature
Apparent ('feels-like') temperature often correlates with demand slightly better than dry-bulb temperature.

In [ ]:
fig = scatter_with_binned_mean("apparent_temperature_mean", "UK mean apparent temperature", "Demand vs apparent temperature")
save_fig(fig, "fig_09_demand_vs_apparent_temp"); plt.show()
print("Correlations with demand:")
print(df[["nd_mw","temperature_mean","apparent_temperature_mean"]].corr()["nd_mw"].round(3))


## 10. Missing-value heatmap
Using the `nd_mw_is_missing` / `weather_is_missing` flags produced by `clean.py`. Shows *where* gaps concentrate rather than assuming they are uniform.

In [ ]:
miss = df.pivot_table(index="year", columns="month", values="nd_mw_is_missing", aggfunc="sum", fill_value=0)
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(miss.values, aspect="auto", cmap="Reds")
ax.set_xticks(range(miss.shape[1])); ax.set_xticklabels(miss.columns)
ax.set_yticks(range(miss.shape[0])); ax.set_yticklabels(miss.index)
ax.set_title("Missing demand half-hours by year/month"); ax.set_xlabel("Month"); ax.set_ylabel("Year")
fig.colorbar(im, ax=ax, label="missing count")
save_fig(fig, "fig_10_missing_heatmap"); plt.show()
print("Total missing demand slots:", int(df['nd_mw_is_missing'].sum()))
print("Total missing weather slots:", int(df['weather_is_missing'].sum()))


## 11. Outlier investigation
Robust (median/MAD) z-score on demand. Flag |z|>6 and inspect — genuine extremes (cold snaps) vs. data errors. Note 2020 lockdown context.

In [ ]:
s = demand.dropna()
med = s.median(); mad = (s - med).abs().median()
robust_z = 0.6745 * (s - med) / (mad if mad else 1)
outliers = s[robust_z.abs() > 6]
print(f"{len(outliers)} candidate outliers (|robust z| > 6)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(s, bins=80, color="steelblue"); axes[0].set_title("Demand distribution"); axes[0].set_xlabel("MW")
axes[1].plot(s.index, s.values, lw=0.2, color="grey")
axes[1].scatter(outliers.index, outliers.values, color="red", s=10, label="outliers")
axes[1].axvspan(pd.Timestamp("2020-03-23", tz="UTC"), pd.Timestamp("2020-06-15", tz="UTC"),
                color="orange", alpha=0.15, label="1st lockdown")
axes[1].set_title("Outliers in context"); axes[1].legend(); axes[1].set_ylabel("MW")
save_fig(fig, "fig_11_outliers"); plt.show()
outliers.sort_values().head(10)


## 12. Autocorrelation and partial autocorrelation
Expect strong spikes at the daily lag (48) and weekly lag (336), confirming the seasonal structure the models must capture.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
s = demand.dropna()
fig, axes = plt.subplots(2, 1, figsize=(13, 9))
plot_acf(s, lags=400, ax=axes[0])
axes[0].set_title("ACF (note spikes at lag 48 = 1 day, 336 = 1 week)")
plot_pacf(s, lags=100, ax=axes[1], method="ywm")
axes[1].set_title("PACF")
save_fig(fig, "fig_12_acf_pacf"); plt.show()


## 13. Seasonal decomposition
STL on a representative ~60-day window (daily period = 48) to separate trend / daily-seasonal / residual. For annual seasonality, decompose the daily-mean series instead.

In [ ]:
from statsmodels.tsa.seasonal import STL
s = demand.dropna()
mid = s.index[len(s)//2]
window = s.loc[mid: mid + pd.Timedelta(days=60)].asfreq("30min").interpolate()
stl = STL(window, period=48, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(13, 9)
fig.suptitle("STL decomposition (daily period), 60-day window", y=1.01)
save_fig(fig, "fig_13_seasonal_decomposition"); plt.show()


## 14. Demand around daylight-saving transitions
Sanity-check the DST handling from `time_utils`: the spring day has **46** half-hours, the autumn day **50**. Demand should be continuous across both.

In [ ]:
per_day = df.groupby("local_date").size()
spring = per_day[per_day == 46].index
autumn = per_day[per_day == 50].index
print("Spring (46-period) days:", [d.date() for d in spring][:5])
print("Autumn (50-period) days:", [d.date() for d in autumn][:5])

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, days, label in [(axes[0], spring, "Spring forward (46)"), (axes[1], autumn, "Autumn back (50)")]:
    if len(days):
        d = days[len(days)//2]
        w = demand.loc[d - pd.Timedelta(days=1): d + pd.Timedelta(days=2)]
        ax.plot(w.index, w.values, color="purple")
        ax.axvline(d, color="black", ls="--", alpha=0.6)
        ax.set_title(f"{label}: {d.date()}")
    ax.set_ylabel("MW")
save_fig(fig, "fig_14_dst_transitions"); plt.show()


## Strongest figures for the README

Feature only these 4–5 (not every experimental graph):

1. `fig_01_demand_timeseries.png` — the whole series, incl. any 2020 shift.
2. `fig_05_halfhour_profile.png` — daily seasonality (the dominant cycle).
3. `fig_04_demand_by_dayofweek.png` — weekly seasonality.
4. `fig_08_demand_vs_temperature.png` — the non-linear temperature relationship.
5. `fig_13_seasonal_decomposition.png` — trend + seasonal + residual at a glance.

The rest (`fig_02`, `03`, `06`, `07`, `09`, `10`, `11`, `12`, `14`) stay in the
notebook / `reports/figures/` as supporting evidence.
